# Python中級編5: ファイルの読み書き（2）


## 目次
* [課題14.1: 画像の圧縮](#課題14.1:-画像の圧縮)
 * [1. ファイルの読み込み](#1.-ファイルの読み込み)
 * [2. ランレングス符号化](#2.-ランレングス符号化)
 * [3. 圧縮結果の読み込みと展開](#3.-圧縮結果の読み込みと展開)
* [課題14.2: 音声データの圧縮](#課題14.2:-音声データの圧縮)
 * [1. ファイルの読み込み](#1.-ファイルの読み込み)
 * [2. 音声データを見る・聞く](#2.-音声データを見る・聞く)
 * [3. 差分圧縮](#3.-差分圧縮)
 * [4. 非可逆圧縮](#4.-非可逆圧縮)
  * [4.1 ダウンサンプリング](#4.1-ダウンサンプリング)
  * [4.2 再量子化](#4.2-再量子化)

---
**始める前の注意**：

今回のノートブックは
```python
!ls -l 
```
のようなターミナルコマンド（シェルコマンド）を実行するセルが沢山あります。
ターミナルコマンドとは、上の例では `ls -l` の部分で、その名の通りターミナル上で打ち込めば実行できるコマンド（小さなプログラム）です。
Jupyter Notebook の中からターミナルコマンドを実行する場合は、行の先頭に `!` を書き、そのあと続けてターミナルコマンドを書きます。

<font color="red">このノートブックの中で使っているターミナルコマンドは Mac 用です。普段は Windows で課題をやっている人も今回は Mac でやってください</font>

---

このノートブックでは、前回に続き、「ファイル入力」と「ファイル出力」の部分まで含めたプログラミングの練習を行う。

始める前に、LETUS の今回のセクションから以下のファイルをダウンロードし、このノートブックと同じフォルダに置きなさい。
* image.txt
* speech.txt

## 課題14.1: 画像の圧縮

このノートブックと一緒に配布してあるテキストファイル `image.txt` はある画像を表すデータだが、白のピクセルを `1` 、黒のピクセルを `0` としてテキスト形式で画像を表している。

まずこのファイルが何バイトあるか確認せよ：

In [ ]:
!ls -l image.txt

およそ4.5メガバイトつまり450万バイトほどあることが分かる。

この問題の目的は、画像としての内容を壊さずに（復元可能なように）このデータを圧縮することである。

### 1. ファイルの読み込み

ファイルの各行は画像の横のラインを表しており、ピクセルを表す `1` と `0` は各行中で
```
1111100011001010001
```
のようにすきまなく並べられている。

まずこのファイルを読み込み、<font color="red">文字列ではなく</font>整数の 1 と 0 を並べた2重リストにせよ。

つまり、もしファイル `in_file_name` の内容が
```
11011
00100
10101
```
だったとしたら
```python
[[1,1,0,1,1],
 [0,0,1,0,0],
 [1,0,1,0,1]]
```
という2重リストを作成して返す関数 `read_image(in_file_name)` を実装せよ。

注意：実際のファイルの内容が何行・何列であっても対応できるように実装せよ。各行の文字数は同じであると仮定してよい。

ヒント（ダブルクリックで表示）
<!--
以下のように実装できるだろう

* まず２重リスト全体を表す変数 img を img = [] と初期化する

* ファイルの各行に対して以下を行う
    * ファイルを1行読み込み, 変数 line に入れる

    * 空のリスト row を作る

    * i = 0, 1, ..., len(line)-1 に対して
         line の i 文字目が "1" だったら row に整数の 1 を追加する
         line の i 文字目が "0" だったら row に整数の 0 を追加する

    * row を img に追加する

readline で読み込んだ文字列の最後には改行文字 \n が含まれることに
注意せよ（改行文字に対応する余計な 1 や 0 を加えないように注意せよ）
-->

In [ ]:
def read_image(in_file_name):
    # *** 実装せよ ***


実装できたら、まず小さなデータでテストしなさい：

In [ ]:
# 3行5列の小さい画像データの作成
!echo "11011" > test-mini-image.txt
!echo "00100" >> test-mini-image.txt
!echo "10101" >> test-mini-image.txt

# 小さい画像データの中身を表示
!cat test-mini-image.txt

In [ ]:
mini_img = read_image("test-mini-image.txt")
print("読み込み結果:", mini_img)

mini_img == [[1,1,0,1,1],
             [0,0,1,0,0],
             [1,0,1,0,1]]

正しく実装できていそうだったら、以下のセルを実行して2重リストの形式で `image.txt` を読み込みなさい：

In [ ]:
img = read_image("image.txt")

一部を取り出して表示してみよう。

以下のセルを実行すると，画像から 30 x 30 ピクセルを切り出したものが表示される：

In [ ]:
for y in range(600, 630):
    print(img[y][800:830])

何の画像だかよくわからないであろう。以下のセルを実行すると画像全体を白黒画像として表示する（このセルの中身じたいを理解する必要はない）：

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 15))
plt.imshow(img, cmap=plt.cm.gray);

やはり何の画像だか分からない場合は `read_image` の実装が間違っている。

画像の出典は[このページ](https://en.wikipedia.org/wiki/File:De_Alice%27s_Abenteuer_im_Wunderland_Carroll_pic_04.jpg)である。

画像が読み込めた人は次へ進もう。

### 2. ランレングス符号化
画像を見ると分かるように、白いピクセルが連続する部分あるいは黒いピクセルが連続する部分が非常に多いことが分かる。

この性質を利用して、画像の各行（横のライン）を以下のように圧縮してみよう。

例えば画像のある行が（リストとして）
```python
[1, 1, 1, 0, 0, 1, 1, 1, 1]
```
となっていたとする。これを
```
3,2,4
```
と表す。

これはラインの中で同じ数字が連続するところをひとかたまりと考え、上の例では
```
1が3つ、0が2つ、1が4つ
```
となることから
```
最初に連続する1の数, 次に連続する0の数, その次に連続する1の数,...
```
という風に、同じ数字の連続する数をカンマ区切りで表したものである。
カンマの前後には空白を入れないことに注意せよ。

各行はかならず「最初に連続する1の数」から始めるものとする。
つまり、実際の最初の数字が 0 だった場合は、「1が0個連続したもの」が先頭にあると考え、例えばある行が
```python
[0, 0, 1, 1, 1, 0]
```
となっていたら
```
0,2,3,1
```
と表すことにする。

最初の数字は（0個でも）「1」だと決めたので、以降、1, 0のどちらが何個つづくかはこの表現から完全に復元できる。

このような圧縮方法を「ランレングス符号化」という。

それでは、画像を表す2次元リスト `image` と、出力ファイル名 `out_file_name` を受け取り、
上記の方法で各行を圧縮した結果を（行ごとに）ファイルに書き出す関数 `encode_image(image, out_file_name)` を実装しなさい。

例えば `image` が
```python
[[1,1,0,1,1],
 [0,0,1,0,0],
 [1,0,1,0,1]]
```
という２重リストならば、`out_file_name` の内容は
```
2,1,2
0,2,1,2
1,1,1,1,1
```
となるはずである。


ヒント：まず画像の行を表すリスト、例えば `[1,1,0,1,1]` を受け取って、そのランレングス符号化の結果を `[2,1,2]` のように整数のリストとして返す関数 `encode_row` を実装するとよい。符号化とファイルへの書き出しをいっぺんにやろうとすると複雑になって大変である。

なので、まず下のセルの `def encode_image(...)`  の前に自分で `encode_row` を定義し、その次に、それを使って `encode_image` を実装せよ。

ヒント（ダブルクリックで表示）
<!--
encode_row を使えば encode_image(image, out_file_name) は
以下のように実装できる：

* 出力ファイルを open する
* image の各行について
   * 行を encode_row で連続する1/0の長さのリストに変換する
   * 変換後のリストの内容を、カンマ区切りでファイルに書き出す

[2,1,2] のようなリストをカンマ区切りで書き出すには

print(最初の要素,  file=出力ファイル, end="") # 改行せずに最初の要素を書く
print(",", file=出力ファイル, end="")       # つづけてカンマを書く・改行しない
print(2番目の要素, file=出力ファイル, end="") # つづけて2番目の要素を書く・改行しない
print(",", file=出力ファイル, end="")       # つづけてカンマを書く・改行しない
...
print(最後の要素, file=出力ファイル, end="") # つづけて最後の要素を書く・改行しない
print(file=出力ファイル) # ただ改行するだけ

という手続きを for ループと if 文を組み合わせて実行すればよい。
-->

In [ ]:
# ここに encode_row を定義する
# 引数:
#   image: 要素が 1, 0 の２重リストで表した画像
#   out_file_name: 出力ファイル名
def encode_image(image, out_file_name):
    # *** 実装しなさい ***


実装できたら、まず小さなデータでテストしなさい：

In [ ]:
# 小さい画像データを読み込み
mini_img = read_image("test-mini-image.txt")

# 符号化の結果をファイルに保存
encode_image(mini_img, "test-mini-image-encoded.txt")

In [ ]:
# 入力ファイルの内容を表示
!echo "-------"
!echo "入力画像"
!echo "-------"
!cat test-mini-image.txt
# 出力ファイルの内容を表示
!echo "----------"
!echo "出力ファイル"
!echo "----------"
!cat test-mini-image-encoded.txt

小さいデータのでのテスト結果が正しそうだったら、大きな画像の圧縮結果をファイル `image-encoded.txt` に保存しなさい：

In [ ]:
encode_image(img, "image-encoded.txt")

圧縮結果のファイルの大きさを見てみなさい：

In [ ]:
!ls -l image-encoded.txt

正しく実装できていれば約360キロバイト（= $360 \times 10^3 = 360,000$バイト）程度の大きさ、つまり元画像の1/10以下になっているはずである。

### 3. 圧縮結果の読み込みと展開
次に圧縮結果を元の画像に戻せることを確認しよう。

圧縮結果を保存したファイル名 `enc_file_name` を受け取り、画像データを 1/0 の2重リストの形に戻して返す関数 `decode_image(enc_file_name)` を実装しなさい。

ヒント１：ランレングス符号化によって圧縮した1行分のデータ、つまり `"2,1,2"` のような文字列 `row` を受け取って、画像の1行分のデータを`[1,1,0,1,1]` のような 1/0 のリストに変換して返す関数 `decode_row(row)` をまず実装しなさい。

ヒント２：カンマ区切りの `"2,1,2"` のような文字列 `s` を整数のリストにするには、まず `s.split(",")` で `s` をカンマで切った文字列のリストを作る。つまり `["2", "1", "2"]` のようなリストを作り、次にその各要素を `int` で整数に変換すればよい。

ヒント３：ゼロが `n` 個つづくリストを作るには `[0] * n` とすればよい。

In [ ]:
#-----
# まず decode_row を実装しなさい
#-----
def decode_row(row):
#-----
# 次に decode_row を使って decode_image(enc_file_name) を実装しなさい
#-----
# 引数:
#   enc_file_image: ランレングス符号化の結果を保存したファイル名
def decode_image(enc_file_name):
    # *** 実装しなさい ***


実装できたら、まずは先ほど作った小さいテスト結果を読み込んで2次元リストに戻してみよう：

In [ ]:
decode_image("test-mini-image-encoded.txt")

元の3行5列の画像
```
11011
00100
10101
```
に対応した2重リスト、つまり
```python
[[1, 1, 0, 1, 1], [0, 0, 1, 0, 0], [1, 0, 1, 0, 1]]
```
が表示されていれば正しく実装できていそうである。

では元の大きな画像の圧縮結果を読み込んで画像として表示してみよう：

In [ ]:
# ランレングス符号化を元に戻す
decoded = decode_image("image-encoded.txt")

# 画像として表示
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 15))
plt.imshow(decoded, cmap=plt.cm.gray);

元のファイルを読み込んだ結果と完全に一致することも確かめよう：

In [ ]:
# 元のデータ
orig_img = read_image("image.txt")

decoded == orig_img

<!-- 
元データ作成

from PIL import Image
import numpy as np
filename = 'alice.jpg'

# 画像ファイルパスから読み込み
img = Image.open(filename)

# numpy配列の取得
img_array = np.asarray(img)

bw = []
thresh = 600
for row in img_array:
    bw_row = []
    for px in row:
        if sum(px) < thresh:
            bw_row.append(0)
        else:
            bw_row.append(1)
    bw.append(bw_row)
    
    
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 15))
plt.imshow(bw, cmap=plt.cm.gray)

with open("alice-img.txt", "w") as f:
    for row in bw:
        print(*row, sep="", file=f)   
-->

## 課題14.2: 音声データの圧縮

このノートブックと一緒に配布しているファイル `speech.txt` はある音声データをテキストファイル形式で保存したものである。

このセクションでは、音声データを圧縮するプログラムを題材にしてファイル読み書きとデータの加工の練習をする。

### 1. ファイルの読み込み

`speech.txt` は各行に整数をひとつづつ書いた
```
100
120
-80
-205
25
...
```
という形式になっている。

**練習**: 上のような、1行にひとつの整数が書かれたファイル `file_name` を読み込み、各行の整数を行の順にリストとして並べたものを返す関数 `read_speech_data(file_name)` を実装しなさい。

例えば `file_name` の内容が
```
100
110
130
```
という3行だったら、`read_speech_data(file_name)` の返り値は
```python
[100, 110, 130]
```
というリストになる。

`readline` で読み込んだデータは**常に文字列**であることに注意しなさい。例えば上の3行のファイルの例では、最初の行を `readline` で読み込んだ結果は `"100\n"` という文字列であって、整数の `100` ではない。`"100\n"` を整数に変換するには `int` 関数を用いればよい（最後の `"\n"` は `int` 関数では無視されて、`"100"` の部分が整数に変換される）

In [ ]:
def read_speech_data(file_name):
    # *** 実装しなさい ***


ファイルの行数はターミナルコマンド `wc -l` で確認できる。"speech.txt" が何行あるか確認しなさい：

In [ ]:
!wc -l speech.txt

`read_speech_data` で読み込んだ結果のリストの長さが `wc -l` で表示される行数と一致していることを確認しなさい：

In [ ]:
len(read_speech_data("speech.txt"))

### 2. 音声データを見る・聞く
データを加工して圧縮する前にどういうデータか見てみよう。

下のセルを実行すると、読み込んだ "speech.txt" の内容がグラフとして表示される：

In [ ]:
import matplotlib.pyplot as plt

samples = read_speech_data("speech.txt")

plt.figure(figsize=(12, 4))
plt.plot(samples);

このグラフは "speech.txt" のデータの行番号を横軸、各行の数値を縦軸にとってグラフとして表示している。

一部を拡大してみると、波のような形のデータであることがわかる：

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(samples[2000:2500]);

"speech.txt" はある音声を一秒間に16000回記録したデータである。すなわち、1/16000 秒ごとに音声の振幅値を記録したものが "speech.txt" の各行の値である。

各時点での振幅は16ビットの符号つき整数つまり $-2^{15} = -32,768$ から $2^{15}-1 = 32,767$ の範囲の整数として記録されている。

データを音声として聞いてみよう：

In [ ]:
import IPython.display

IPython.display.Audio(samples, rate=16000)

この課題は音声データを圧縮することが目的なので、上のセルの音声再生の仕組みについて特に気にする必要はないが、`rate=16000` は「1秒間に16000回振幅を記録した音声データである」ことを指定している。

### 3. 差分圧縮

まず音声の品質を変えない（復元可能）なタイプの圧縮方法を考えよう。

音声データは
```
1000, 1200, 1270, 1070, ...
```
のように前後にある程度の連続性がある（近い値が並んだ）数列になっている。
このことから、隣り合う数値の差分をとったもの（階差数列）は
```
200, 70, -100, ...
```
のように元のデータ列に比べて桁数が小さい数値がより多く出てくる可能性がある。

このアイデアに従って、次のような圧縮方式を実装してみよう。

1. データとして `xs = [100, 120, 90, 70, ...]` のようなリスト `xs` が与えられたとき、同じ長さのリスト `ys` を以下のように作成する
   * `ys` の最初の要素は `xs` の最初の要素と同一: `ys[0] = xs[0]`
   * `i > 0` に対し、`ys[i]` は `xs[i]` と `xs[i-1]` の差分：`ys[i] = xs[i] - xs[i-1]  (i > 0 のとき)`
2. `ys` を "speech.txt" と同じ形式、つまり各行にひとつの整数を書く形式で保存する

上のように作った `ys` からは元の `xs` が完全に復元できることに注意しなさい。

---

それでは、まず入力された整数のリスト `xs` に対し 1. の処理を行い、結果として得られる `ys` をリストとして返す関数 `diff_encode(xs)` を実装しなさい。例えば `xs = [1, 2, -3]` のとき、`diff_encode(xs)` の返り値は `[1, 1, -5]` になる。`xs` として空のリストが入力されることはないものとしてよい。

In [ ]:
def diff_encode(xs):
    # *** 実装しなさい ***


実装できたらテストしなさい：

In [ ]:
diff_encode([1, 2, 2, -3]) == [1, 1, 0, -5]

---
次に、入力された整数のリスト `ys` を "speech.txt" と同じ形式（1行に1要素）で、ファイル名 `out_file_name` のファイルに書き出す関数 `write_speech_data(ys, out_file_name)` を実装しなさい：

In [ ]:
def write_speech_data(ys, out_file_name):
    # *** 実装しなさい ***


実装できたら、`read_speech_dat` で "speech.txt" から読み込んだデータを `diff_encode` で圧縮し、結果を "speech-diff-encoded.txt" というファイル名で書き出しなさい：

In [ ]:
# read_speech_data, diff_encode, write_speech_data
# を使って、音声データの読み込み -> 圧縮 -> 保存の処理を実装しなさい



圧縮結果を書き出したら、`ls -l` コマンドで圧縮前後のファイルサイズの変換を見てみなさい：

In [ ]:
!ls -l speech.txt speech-diff-encoded.txt

10% くらい圧縮できていればおそらく正しく実装できている。

---
次に、上で実装した差分による圧縮結果のデータをリスト `ys` として入力し、元の音声データの形に戻してリストとして返す関数 `diff_decode(ys)` を実装しなさい。`ys` として空リストが渡される可能性はないものとしてよい。

例えば `ys = [100, 10, -5]` のとき `diff_decode(ys)` の返り値は `[100, 110, 105]` となるはずである。

In [ ]:
def diff_decode(ys):
    # *** 実装しなさい ***


実装できたら小さいデータでまずテストしてみなさい：

In [ ]:
diff_decode([100, 10, -5]) == [100, 110, 105]

正しく動いていそうだったら、"speech-diff-encoded.txt" に保存したデータを読み込み、`diff_decode` で元の音声データに戻した結果が最初と一致することを確認しなさい：

In [ ]:
# 元の音声データ
samples = read_speech_data("speech.txt")

# 差分圧縮したデータ
diff_enc = read_speech_data("speech-diff-encoded.txt")

# 差分圧縮を元に戻す
diff_dec = diff_decode(diff_enc)

# 元の音声データと比較
samples == diff_dec       

### 4. 非可逆圧縮

次に、少し音質を犠牲にして、データ量を小さくする方法を実装してみよう。
そのような、完全に元のデータには戻せない（復元できない）ような方法でデータを圧縮することを不可逆圧縮という。

#### 4.1 ダウンサンプリング

まず簡単な方法として、もともと１秒間に16000回記録してあった振幅のデータを「1秒間にその半分つまり8000回しか記録しなかった」ことにしてみる。このようにデータを「間引く」ことをダウンサンプリングという。

つまり、元の音声データのリストを `xs = [10, 20, 30, 40, 50, ...]` とすると `ys = [10, 30, 50, ...]` と `xs` の要素を一つおきで抜き出したデータにしてみよう。これでデータ量はほぼ半分になるはずだ。

**練習**

入力された整数のリスト `xs` に対して、`[xs[0], xs[2], xs[4], ...]` を返す関数 `downsample(xs)` を実装しなさい。

返すリストの長さは `(len(xs) + 1) // 2` とする。つまり `xs` の長さが偶数だったらちょうど半分、奇数だったら (`xs` の長さ + 1) / 2 の長さになるようにする（ひとつおきに振幅値を取っていけば自然にそうなる）。

In [ ]:
def downsample(xs):
    # *** 実装しなさい ***


実装できたらテストしなさい：

In [ ]:
print(downsample([10, 20, 30, 40, 50, 60]) == [10, 30, 50])
print(downsample([10, 20, 30, 40, 50]) == [10, 30, 50])

正しく動いていそうだったら、`write_speech_data` を使って "speech-downsample.txt" というファイル名で `downsample` の結果を保存しなさい：

In [ ]:
# 元の音声データ
xs = read_speech_data("speech.txt")

# downsample で xs を半分に圧縮し、ファイル "speech-downsample.txt" に結果を保存しなさい

# ** 自分で書く **


保存したら、`ls -l` コマンドでどれくらい圧縮できたか見てみなさい：

In [ ]:
!ls -l speech.txt speech-downsample.txt

約半分のファイルサイズになっているはずである。

半分にダウンサンプリングした結果を元の音声と比べて聞いてみよう。

プログラムが間違っているとひどい音がする場合がある。
念のため最初は小さな音量で聞いてみなさい：

In [ ]:
import IPython.display

# 元の音声データを読み込み
original = read_speech_data("speech.txt")

# ダウンサンプリングしたデータを読み込み
downsampled = read_speech_data("speech-downsample.txt")

# 1秒間に 8000 回 振幅を記録したデータとして聞く
print("ダウンサンプリング結果")
IPython.display.display(IPython.display.Audio(downsampled, rate=8000))

print("元の音声データ")
IPython.display.display(IPython.display.Audio(original, rate=16000))

ダウンサンプリングによって音質が悪くなっているのがわかるだろう。

#### 4.2 再量子化
さらにデータを圧縮するために、振幅の大きさを表す数値を16ビットすなわち $2^{16} = 65536$ 段階から、8ビットすなわち $2^{8} = 256$ 段階に粗くしてみよう。

音声データ（振幅）はもともとは連続値なので、$2^{16}$ 段階の整数値として記録した時点で「とびとびの値」で近似したことになる。これを「量子化（quantize）」という。

ここでやろうとしているのは、$2^{16}$段階で近似的に表現された連続データを$2^{8}$段階へと「さらにとびとびの値」で近似することで、これを「再量子化」という。

「再量子化」は単に振幅を小さく（＝音量を小さく）しているのではなく、音声の波のカーブの「なめらかさ」を犠牲にしてデータサイズを小さくする操作だということに注意してください。

振幅は負の値を取る場合もあるから 16ビットでは -32,768 から 32,767 までの整数値、8ビットでは -128 から 127 までの値を取る。

16ビットから8ビットへの再量子化は、$2^{16} / 2^{8} = 2^8$ 個の16ビットの整数が1つの8ビット整数に対応するように均等に分ければよいので、対応は以下のようになる：

| 16ビットの振幅値の範囲| 8ビットに再量子化した振幅値 |
|:------------------:|:-------------:|
|$127 \times 2^{8}$ 〜 $128 \times 2^{8}-1$|          127|
|$126 \times 2^{8}$ 〜 $127 \times 2^{8}-1$|          126|
|...                |         ...|
|$0$ 〜 $2^{8}-1$   |           0|
|$-2^{8}$ 〜 $-1$    |           -1|
|...                |         ...|
|$-127 \times 2^{8}$ 〜 $-126\times 2^{8}-1$|-127|
|$-128 \times 2^{8}$ 〜 $-127\times 2^{8}-1$|-128|

これは要するに16ビットの振幅値を 〇〇 で割ったもの（小数点以下は切り捨て）を8ビットの振幅値とすればよい、ということである。〇〇のところは自分で考えてみなさい。

では、16ビット整数で記録された音声データをリストで表した `xs` を入力とし、`xs` の各要素を8ビットで再量子化したデータのリストを返す関数 `quantize_8bit(xs)` を実装しなさい：

In [ ]:
def quantize_8bit(xs):
    # *** 実装しなさい ***


実装できたらテストしましょう：

In [ ]:
res = quantize_8bit([-32768, -32641, -32512, -2, -1, 0, 1, 127, 255, 256, 512, 32767])
print("再量子化結果：", res)

res == [-128, -128, -127, -1, -1, 0, 0, 0, 0, 1, 2, 127]

正しく動いていそうだったら
* 元のデータを8ビットで再量子化したデータをファイル "speech-8bit.txt" に保存しなさい
* 元のデータに `downsample` を適用し、さらに8ビットで再量子化したデータを "speech-downsample-8bit.txt" に保存しなさい
* 元のデータに `downsample` を適用し、さらに8ビットで再量子化した上で `diff_encode` によって差分圧縮したデータを "speech-downsample-8bit-diff.txt" に保存しなさい

In [ ]:
# ファイル "speech-8bit.txt" 
# ファイル "speech-downsample-8bit.txt" および
# ファイル "speech-downsample-8bit-diff.txt" を上の通り作成しなさい


下のセルを実行し、`ls -l` コマンドで以下のファイルのサイズを確認しなさい
* 元データ（`speech.txt`）
* 8bit に再量子化した結果（`speech-8bit.txt`）
* 半分にダウンサンプリングした結果（`speech-downsample.txt`）
* 半分にダウンサンプリングしてから 8bit に再量子化した結果（`speech-downsample-8bit.txt`）
* さらに差分圧縮した結果（`speech-downsample-8bit-diff.txt`）

In [ ]:
!ls -l speech.txt 
!ls -l speech-8bit.txt 
!ls -l speech-downsample.txt 
!ls -l speech-downsample-8bit.txt
!ls -l speech-downsample-8bit-diff.txt

ダウンサンプリング＋再量子化＋差分圧縮によって元ファイルの 1/4 程度のサイズになっているはずである。

最後に、再量子化・ダウンサンプリング・再量子化＋ダウンサンプリングのそれぞれで音質がどう変わるか確認してみよう。

ヘッドホンかイヤホンで聞くと違いが分かりやすいだろう（音量に気をつけて）。

In [ ]:
import IPython.display

# 元の音声データを読み込み
original = read_speech_data("speech.txt")

# 8 bit に再量子化したデータを読み込み
quant_8bit = read_speech_data("speech-8bit.txt")

# downsample したデータを読み込み
downsampled = read_speech_data("speech-downsample.txt")

# downsample した上で 8bit に再量子化したデータを読み込み
downsampled_8bit = read_speech_data("speech-downsample-8bit.txt")

print("元の音声データ")
IPython.display.display(IPython.display.Audio(original, rate=16000))

print("8ビットに再量子化")
IPython.display.display(IPython.display.Audio(quant_8bit, rate=16000))

print("ダウンサンプリング")
IPython.display.display(IPython.display.Audio(downsampled, rate=8000))

print("ダウンサンプリング + 8ビットに再量子化")
IPython.display.display(IPython.display.Audio(downsampled_8bit, rate=8000))

---
お疲れ様でした。今学期の課題はこれで全て終わりです。

- Run -> Run All Cells を実行して保存のボタンを押し、<font color="red">全ての実行結果を保存した上で</font>提出してください。
- 実行結果が保存されていない問題は<font color="red">採点しません</font>。